# Browser Automation for whosampled.com

## Browser automation is no longer working.
For the Hip-Hop Time Travelers substack, we need to get samples for used for each album. We are using the website whosampled.com. The thing is, they have their website securely locked down. Web scrapping does not work. And even browser automation will get caught eventually within a couple hours of starting. So be quick (but not too quick!) and dilligent.

The first album to be featured on *Hip-Hop Time Travelers* is **Like Water For Chocolate**. For future album features, **replace the album, artist, and top track name** so that the code can navigate to the respective album's webpage. Navigating directly to the album's webpage e.g. https://www.whosampled.com/album/Common/Like-Water-For-Chocolate could flag Cloudflare to block the IP address.

Album: Like Water For Chocolate
Artist: Common
Released: March 28, 2000

Scrap whosampled.com and get
- song titles
- samples

Clean the data and save to csv

In [4]:
import os
import json
import random
import time

In [ ]:
from playwright.async_api import async_playwright, expect, Keyboard

In [5]:
script_dir = os.path.dirname(os.path.abspath('get-samples.ipynb'))

In [6]:
with open('consts.json', 'r') as file:
    data = json.load(file)

album = data['album']
artist = data['artist']

track = "The Light"

Browser automation steps that no longer work. However, maybe one day, they will work again...

In [9]:
user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36'

In [22]:
async def open_browser(headless=False, user_agent=user_agent):
    """
    Starts the automated browser and opens a new window
    """
    # Start playwright
    playwright = await async_playwright().start()

    # Open firefox browser, can use chromium (chrome) or others
    browser = await playwright.chromium.launch(headless=False)

    # set a user agent
    context = await browser.new_context(user_agent=user_agent)
  
    # Create a new browser window
    page = await browser.new_page()

    return browser, page

In [23]:
driver, page = await open_browser()

In [24]:
url = 'https://www.whosampled.com/'
await page.goto(url)

<Response url='https://www.whosampled.com/' request=<Request url='https://www.whosampled.com/' method='GET'>>

In [9]:
xpath_search = '//input[@aria-label="Type any track, artist, movie or TV show"]'
search = page.locator(xpath_search)

In [10]:
await search.click()

In [16]:
await search.fill(artist)

In [17]:
await page.keyboard.press('Enter')

In [27]:
xpath_artist_page_link = '//div[@class="topHit"]//a[@class="artistName"]'
artist_page_link = page.locator(xpath_artist_page_link)

In [28]:
await artist_page_link.click()

In [33]:
xpath_top_song_link = f'//span[text()="{track}"]/parent::a'
top_song_link = page.locator(xpath_top_song_link)

In [34]:
await top_song_link.click()

In [35]:
xpath_album_link = '//div[@class="release-name"]/a'
album_link = page.locator(xpath_album_link)

In [36]:
await album_link.click()

In [38]:
# Save the album's track list page
input_file_path = os.path.join(script_dir, '..', 'data', f'{album}', 'input', f'track_list_{album}.html')

source = await page.content()
with open(input_file_path, 'w') as f:
    f.write(source)

In [7]:
xpath_track_list = '//section[@class="trackList bordered-list"]//section'

xpath_track_name = './/h3[@class="trackName"]//span[@itemprop="name"]//text()'

xpath_track_name_link = './/h3[@class="trackName"]//a[@itemprop="url"]//@href'

xpath_samples = './/span[normalize-space(text()) = "sampled"]//following-sibling::ul//li//text()'

In [ ]:
import asyncio

In [ ]:
# Click into each track so that we can save each page source
track_list = page.locator(xpath_track_list)

for track in track_list:    
    track_name_link = track.xpath(xpath_track_name_link)
    track_name = track.xpath(xpath_track_name)

    await track_name_link.click()
    source = await page.content()

    input_file_path = os.path.join(script_dir, '..', 'data', f'{album}', 'input', f'track_details_{track_name}.html')
    
    with open(input_file_path, 'w') as f:
        f.write(source)
        await asyncio.sleep(5)
    

In [26]:
await driver.close()

Because browser automation no longer works, because Cloudflare sucks and because Spotify sucks, we have to manually download each track's details page. It will be a pain in the arse however it is the only way now since this website is quite locked-down.

Manually save each track's details to `data/{album}/input` as `track_details_{track_name}.html` then proceed below.

We will get credits, samples, sampled in, remixed, and covered lists.

In [22]:
import glob
import pandas as pd
from lxml import etree, html

In [8]:
# list the scraped pages to parse them
input_file_path = os.path.join(script_dir, '..', 'data', f'{album}', 'input', f'track_details*.html')
files = glob.glob(input_file_path)

Step through these functions to see if the output works with the new way to get sample data

To do:
Get sampled in, covered, remixed

In [63]:
def parse_sample_list(raw_list):
    """
    Parse a flat list like:
    ['', 'Title', 'Artist', 'Year', 'Sample Type', '', 'Title', ...]
    into a list of dicts: {sample_title, sample_artist, sample_year, sample_type}
    """
    records = []
    i = 0
    n = len(raw_list)

    while i < n:
        if raw_list[i] == "":
            title, artist, year, sample_type = raw_list[i + 1 : i + 5]
            records.append({
                "sample_title": title,
                "sample_artist": artist,
                "sample_year": year,
                "sample_type": sample_type,
            })
            i += 5
        else:
            i += 1
    
    return records

In [ ]:
xpath_samples_table = '//h2[text() = "Song Connections"]//following-sibling::section//h3[contains(text(), "samples")]/following::*[1]//tbody'
xpath_track_name = '//nav[@class="breadcrumb-wrapper"]//ol//li//meta[@content="2"]//preceding-sibling::span//text()'

In [64]:
data = []
for fn in files:
    tree = html.fromstring(open(fn).read())
    track_name = tree.xpath(xpath_track_name)

    samples_table = tree.xpath(xpath_samples_table)[0]

    rows = samples_table.xpath('.//tr')

    sample_list = []
    for row in rows:
        tds = row.xpath('.//td')
        for td in tds:
            sample_data = td.xpath('normalize-space()')
            sample_list.append(sample_data)

    parsed = parse_sample_list(sample_list)

    for sample in parsed:
        data.append({
            "track_name": track_name[0],
            **sample
        })
data

[{'track_name': 'The Questions',
  'sample_title': "Got 'Til It's Gone",
  'sample_artist': 'Janet Jackson feat. Q-Tip and Joni Mitchell',
  'sample_year': '1997',
  'sample_type': 'Vocals / Lyrics'},
 {'track_name': 'The Light',
  'sample_title': 'Open Your Eyes',
  'sample_artist': 'Bobby Caldwell',
  'sample_year': '1980',
  'sample_type': 'Multiple Elements'},
 {'track_name': 'The Light',
  'sample_title': "You're Getting a Little Too Smart",
  'sample_artist': 'Detroit Emeralds',
  'sample_year': '1973',
  'sample_type': 'Drums'},
 {'track_name': 'The Light',
  'sample_title': 'Track 3 (Another Batch)',
  'sample_artist': 'J Dilla',
  'sample_year': '1998',
  'sample_type': 'Multiple Elements'},
 {'track_name': 'Thelonius',
  'sample_title': 'Vulcan Mind Probe',
  'sample_artist': 'George Duke',
  'sample_year': '1982',
  'sample_type': 'Multiple Elements'},
 {'track_name': 'Thelonius',
  'sample_title': 'Space Intro',
  'sample_artist': 'Steve Miller Band',
  'sample_year': '1976

In [59]:
df = pd.DataFrame(data)
df

,track_name,sample_title,sample_artist,sample_year,sample_type
0,The Questions,Got 'Til It's Gone,Janet Jackson feat. Q-Tip and Joni Mitchell,1997,Vocals / Lyrics
1,The Light,Open Your Eyes,Bobby Caldwell,1980,Multiple Elements
2,The Light,You're Getting a Little Too Smart,Detroit Emeralds,1973,Drums
3,The Light,Track 3 (Another Batch),J Dilla,1998,Multiple Elements
4,Thelonius,Vulcan Mind Probe,George Duke,1982,Multiple Elements
5,Thelonius,Space Intro,Steve Miller Band,1976,Multiple Elements
6,Thelonius,Michelle,The Singers Unlimited,1972,Vocals / Lyrics
7,Payback Is a Grandmother,S.U.S.,Placebo (Jazz Band),1974,Multiple Elements
8,Payback Is a Grandmother,The Payback,James Brown,1973,Vocals / Lyrics
9,Payback Is a Grandmother,Right on Time,Maze Featuring Frankie Beverly,1983,Hook / Riff


In [61]:
output_file_path = os.path.join(script_dir, '..', 'data', f'{album}', 'output', f'samples_{album}.csv')

In [62]:
df.to_csv(output_file_path, index=False)